# exp_001 — Context measurement

This notebook loads recorded trial JSONL, regenerates the processed summary, audits coverage, and plots measured position/context/system curves. It never invents missing cells or treats fixture smoke output as a Qwen finding.

In [ ]:
import csv
import json
import sys
from pathlib import Path

ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))

from llm_lab.analysis import (
    aggregate_jsonl,
    effective_context_by_task,
    effective_context_by_task_and_position,
    missing_context_cells,
    position_curve_rows,
    write_summary_csv,
)

RESULTS_DIR = ROOT / 'experiments/exp_001-context_measurement/results'
RAW_PATH = RESULTS_DIR / 'raw/smoke-trials.jsonl'
SUMMARY_PATH = RESULTS_DIR / 'processed/summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifests/smoke.json'

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if manifest.get('scorer_version') != 'calibrated.v1':
    raise ValueError('manifest uses a non-calibrated scorer policy')
summary_rows = aggregate_jsonl(RAW_PATH, expected_scorer='calibrated.v1')
write_summary_csv(SUMMARY_PATH, summary_rows)

required = {
    'task_type', 'condition_id', 'target_context_tokens',
    'requested_evidence_position', 'actual_evidence_position',
    'n', 'completed_n', 'error_n', 'scored_n', 'accuracy',
}
missing_columns = required - set(summary_rows[0]) if summary_rows else required
if missing_columns:
    raise RuntimeError(f'missing processed columns: {sorted(missing_columns)}')

planned_lengths = sorted({row['target_context_tokens'] for row in manifest['coverage']})
planned_positions = sorted({row['requested_evidence_position'] for row in manifest['coverage']})
planned_tasks = sorted({row['task_type'] for row in manifest['coverage']})
missing = missing_context_cells(
    summary_rows,
    context_lengths=planned_lengths,
    evidence_positions=planned_positions,
    task_types=planned_tasks,
)
if missing:
    raise RuntimeError(f'missing planned cells: {missing[:10]}')

print({
    'phase': manifest['phase'],
    'backend': manifest['backend'],
    'raw_sha256': manifest['raw_results_sha256'],
    'summary_rows': len(summary_rows),
    'excluded_cells': len(manifest['excluded_cells']),
})

In [ ]:
coverage = [
    {key: row[key] for key in (
        'task_type', 'target_context_tokens', 'requested_evidence_position',
        'n', 'completed_n', 'error_n', 'scored_n', 'accuracy',
    )}
    for row in position_curve_rows(summary_rows)
]
coverage

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(planned_tasks), figsize=(15, 4), sharey=True)
axes = [axes] if len(planned_tasks) == 1 else axes
for axis, task_type in zip(axes, planned_tasks):
    for context_tokens in planned_lengths:
        points = [
            row for row in summary_rows
            if row['task_type'] == task_type
            and row['target_context_tokens'] == context_tokens
        ]
        points.sort(key=lambda row: row['requested_evidence_position'])
        axis.plot(
            [row['requested_evidence_position'] for row in points],
            [row['accuracy'] for row in points],
            marker='o',
            label=f'{context_tokens:,} tokens',
        )
    axis.set_title(task_type)
    axis.set_xlabel('requested evidence position')
    axis.set_ylim(0, 1.05)
axes[0].set_ylabel('accuracy')
axes[-1].legend(fontsize='small')
fig.suptitle('Figure A — Position curves')
fig.tight_layout()
plt.show()

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4))
for task_type in planned_tasks:
    points = next(
        row['points'] for row in effective_context_by_task(
            [row for row in summary_rows if row['task_type'] == task_type]
        )
        if row['task_type'] == task_type
    )
    axis.plot(
        [point['context_tokens'] for point in points],
        [point['accuracy'] for point in points],
        marker='o',
        label=task_type,
    )
axis.set_xscale('log', base=2)
axis.set_xlabel('target context tokens')
axis.set_ylabel('accuracy')
axis.set_ylim(0, 1.05)
axis.set_title('Figure B — Context degradation')
axis.legend()
fig.tight_layout()
plt.show()

In [ ]:
effective = effective_context_by_task(summary_rows)
[{
    key: row[key] for key in (
        'task_type', 'baseline_accuracy', 'threshold_accuracy', 'status',
        'crossing_context_tokens', 'effective_context_tokens',
        'largest_tested_context_tokens',
    )
} for row in effective]

In [ ]:
position_sensitive_effective = effective_context_by_task_and_position(summary_rows)
[{
    key: row[key] for key in (
        'task_type', 'evidence_position', 'baseline_accuracy',
        'status', 'crossing_context_tokens', 'effective_context_tokens',
    )
} for row in position_sensitive_effective]

In [ ]:
systems = [
    {
        'task_type': row['task_type'],
        'target_context_tokens': row['target_context_tokens'],
        'median_ttft_s': row['median_ttft_s'],
        'median_prefill_tokens_per_second': row['median_prefill_tokens_per_second'],
        'median_decode_tokens_per_second': row['median_decode_tokens_per_second'],
        'median_peak_memory_bytes': row['median_peak_memory_bytes'],
    }
    for row in summary_rows
]
systems[:5]